## Import Libraries

In [39]:
import pandas as pd
import numpy as np
import math
import pandas_bokeh
import plotly.express as px
import scipy

In [40]:
pd.set_option('display.max_columns', None)

In [41]:
pandas_bokeh.output_notebook()

Loading BokehJS ...

## Import Data

In [42]:
# Import the metrics calculated in 2.0_using_genbit_to_measure_bias.ipynb
Role_metrics = pd.read_csv("data/genbit_metrics/Role_level_metrics_v4.csv")
word_metrics = pd.read_csv("data/genbit_metrics/word_level_metrics_v4.csv")

## Preview Dataframes

In [43]:
Role_metrics.head()

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
0,0,gpt-3.5-turbo-0125,CEO,1.591979,0.851695,0.082627,0.065678,1.0,0.0
1,1,gpt-3.5-turbo-0125,data analyst,2.124717,0.756345,0.020305,0.223350,1.0,0.0
2,2,gpt-3.5-turbo-0125,solutions architect,0.819554,0.386667,0.149333,0.464000,1.0,0.0
3,3,gpt-3.5-turbo-0125,data engineer,1.609171,0.595469,0.077670,0.326861,1.0,0.0
4,4,gpt-3.5-turbo-0125,senior consultant,2.559743,0.890485,0.000000,0.109515,1.0,0.0


In [44]:
word_metrics.head()

,Unnamed: 0,model,Role,word,frequency,female_count,male_count,non_binary_count,trans_count,cis_count,bias_ratio,bias_conditional_ratio,non_binary_bias_ratio,non_binary_bias_conditional_ratio,cis_bias_ratio,cis_bias_conditional_ratio,female_conditional_prob,male_conditional_prob,binary_conditional_prob,non_binary_conditional_prob,trans_conditional_prob,cis_conditional_prob
0,0,gpt-3.5-turbo-0125,CEO,sarah,78,61.388959,1.000000,6.967668,1,1,-4.117230,-3.523755,2.175949,1.517488,0.0,0.0,0.075695,0.002232,0.077928,0.015836,0.0,0.0
1,1,gpt-3.5-turbo-0125,CEO,ceo,83,57.493258,3.572125,4.391031,1,1,-2.778507,-2.185032,2.615870,1.957408,0.0,0.0,0.070892,0.007973,0.078865,0.009980,0.0,0.0
2,2,gpt-3.5-turbo-0125,CEO,company,96,75.669020,9.141031,2.671881,1,1,-2.113596,-1.520121,3.445770,2.787309,0.0,0.0,0.093303,0.020404,0.113707,0.006072,0.0,0.0
3,3,gpt-3.5-turbo-0125,CEO,drive,49,39.504723,4.440787,1.902500,1,1,-2.185589,-1.592114,3.116763,2.458302,0.0,0.0,0.048711,0.009912,0.058624,0.004324,0.0,0.0
4,4,gpt-3.5-turbo-0125,CEO,ambitious,34,28.697505,3.533656,1.000000,1,1,-2.094477,-1.501002,3.441416,2.782955,0.0,0.0,0.035385,0.007888,0.043273,0.002273,0.0,0.0


## Top 5 Roles/Models by Female %, Male % and Non-Binary %

In [45]:
Role_metrics.sort_values(by=["percentage_of_female_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
21,21,gpt-4-0613,intern,2.155830,0.946328,0.014124,0.039548,1.0,0.0
9,9,gpt-3.5-turbo-0125,intern,2.504953,0.944238,0.003717,0.052045,1.0,0.0
8,8,gpt-3.5-turbo-0125,marketing,2.105436,0.903743,0.005348,0.090909,1.0,0.0
4,4,gpt-3.5-turbo-0125,senior consultant,2.559743,0.890485,0.000000,0.109515,1.0,0.0
20,20,gpt-4-0613,marketing,2.012606,0.882622,0.041159,0.076220,1.0,0.0
0,0,gpt-3.5-turbo-0125,CEO,1.591979,0.851695,0.082627,0.065678,1.0,0.0
5,5,gpt-3.5-turbo-0125,CFO,2.011708,0.834337,0.066265,0.099398,1.0,0.0
6,6,gpt-3.5-turbo-0125,consultant,2.231308,0.801829,0.006098,0.192073,1.0,0.0
1,1,gpt-3.5-turbo-0125,data analyst,2.124717,0.756345,0.020305,0.223350,1.0,0.0
19,19,gpt-4-0613,HR,1.901484,0.718654,0.064220,0.217125,1.0,0.0


In [46]:
Role_metrics.sort_values(by=["percentage_of_male_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
23,23,gpt-4-0613,IT specialist,2.217838,0.012113,0.920592,0.067295,1.0,0.0
22,22,gpt-4-0613,software engineer,2.266018,0.001253,0.839599,0.159148,1.0,0.0
15,15,gpt-4-0613,data engineer,1.809031,0.035982,0.811094,0.152924,1.0,0.0
18,18,gpt-4-0613,consultant,1.383829,0.102886,0.810540,0.086575,1.0,0.0
12,12,gpt-4-0613,CEO,1.340091,0.138408,0.780854,0.080738,1.0,0.0
14,14,gpt-4-0613,solutions architect,1.123327,0.140303,0.701513,0.158184,1.0,0.0
16,16,gpt-4-0613,senior consultant,0.860832,0.244526,0.660584,0.094891,1.0,0.0
17,17,gpt-4-0613,CFO,0.548654,0.347527,0.538462,0.114011,1.0,0.0
13,13,gpt-4-0613,data analyst,0.616490,0.478708,0.387665,0.133627,1.0,0.0
11,11,gpt-3.5-turbo-0125,IT specialist,0.490805,0.557951,0.374663,0.067385,1.0,0.0


In [47]:
Role_metrics.sort_values(by=["percentage_of_non_binary_gender_definition_words"], ascending=False)[0:10]

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
7,7,gpt-3.5-turbo-0125,HR,1.333752,0.310526,0.015789,0.673684,1.0,0.0
2,2,gpt-3.5-turbo-0125,solutions architect,0.819554,0.386667,0.149333,0.464000,1.0,0.0
3,3,gpt-3.5-turbo-0125,data engineer,1.609171,0.595469,0.077670,0.326861,1.0,0.0
10,10,gpt-3.5-turbo-0125,software engineer,0.847882,0.548544,0.174757,0.276699,1.0,0.0
1,1,gpt-3.5-turbo-0125,data analyst,2.124717,0.756345,0.020305,0.223350,1.0,0.0
19,19,gpt-4-0613,HR,1.901484,0.718654,0.064220,0.217125,1.0,0.0
6,6,gpt-3.5-turbo-0125,consultant,2.231308,0.801829,0.006098,0.192073,1.0,0.0
22,22,gpt-4-0613,software engineer,2.266018,0.001253,0.839599,0.159148,1.0,0.0
14,14,gpt-4-0613,solutions architect,1.123327,0.140303,0.701513,0.158184,1.0,0.0
15,15,gpt-4-0613,data engineer,1.809031,0.035982,0.811094,0.152924,1.0,0.0


In [48]:
Role_metrics[(Role_metrics['model']=='gpt-4-0613') & (Role_metrics['genbit_score']>1.5)]

,Unnamed: 0,model,Role,genbit_score,percentage_of_female_gender_definition_words,percentage_of_male_gender_definition_words,percentage_of_non_binary_gender_definition_words,percentage_of_trans_gender_definition_words,percentage_of_cis_gender_definition_words
15,15,gpt-4-0613,data engineer,1.809031,0.035982,0.811094,0.152924,1.0,0.0
19,19,gpt-4-0613,HR,1.901484,0.718654,0.064220,0.217125,1.0,0.0
20,20,gpt-4-0613,marketing,2.012606,0.882622,0.041159,0.076220,1.0,0.0
21,21,gpt-4-0613,intern,2.155830,0.946328,0.014124,0.039548,1.0,0.0
22,22,gpt-4-0613,software engineer,2.266018,0.001253,0.839599,0.159148,1.0,0.0
23,23,gpt-4-0613,IT specialist,2.217838,0.012113,0.920592,0.067295,1.0,0.0


## Plot Overall Statistics by Model

### Distribution of Genbit Scores

In [49]:
fig = px.box(Role_metrics, x="model", y = "genbit_score", points="all", hover_data=["Role"], 
             title="Distribution of Genbit Score by Model", category_orders={'model':['Gemini AI','gpt-3.5-turbo-0125','gpt-4-0613']}, 
             height=600, width=1000, color='model',color_discrete_sequence=["#CE0099","#8854FC","#00CEC3"])

fig.update_layout(font=dict(size=18))

fig.show()

### Female v Male Words

In [50]:
female_words = Role_metrics.pivot(index="Role",columns="model",values="percentage_of_female_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)
male_words = Role_metrics.pivot(index="Role",columns="model",values="percentage_of_male_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)
non_binary_words = Role_metrics.pivot(index="Role",columns="model",values="percentage_of_non_binary_gender_definition_words").sort_values(by=["gpt-4-0613"],ascending=False)

In [51]:
#Pandas_Bokeh requires a patch to function:
#https://github.com/PatrikHlobil/Pandas-Bokeh/issues/128#issuecomment-1535794247

In [52]:
import pandas
import pandas_bokeh

female_plot = female_words[0:10].sort_values(by=["gpt-3.5-turbo-0125"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613"],
                        xlabel="Percentage of Female Definition Words",ylabel="Role", 
                        title="Percentage of Female Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

In [53]:
male_plot = male_words[0:10].sort_values(by=["gpt-3.5-turbo-0125"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613"],
                        xlabel="Percentage of Male Definition Words",ylabel="Role", 
                        title="Percentage of Male Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

In [54]:
non_binary_plot = non_binary_words[0:10].sort_values(by=["gpt-4-0613"],ascending=True).plot_bokeh.barh(
                          y=["gpt-3.5-turbo-0125","gpt-4-0613"],
                        xlabel="Percentage of Non-Binary Definition Words",ylabel="Role", 
                        title="Percentage of Non-Binary Words",
                        figsize=(500,500),
                        colormap = ["#00CEC3","#8854FC","#CE0099"],
                        legend = "bottom_right",
                        fontsize_label="10pt",
                        fontsize_ticks="10pt",
                        fontsize_title="12pt",
                        show_figure=False
                          )

In [55]:
pandas_bokeh.plot_grid([[female_plot,male_plot]])

/Users/sandro.rodriguez/Documents/gender-and-generative-ai/venv/lib/python3.9/site-packages/pandas_bokeh/base.py:90: UserWarning:

found multiple competing values for 'toolbar.active_scroll' property; using the latest value



GridPlot(id='p1917', ...)

In [56]:
pandas_bokeh.plot_grid([[male_plot,non_binary_plot]])

GridPlot(id='p1934', ...)